# PesaGuard: Exploratory Data Analysis & Feature Engineering Pipeline

This notebook contains the research phase for **PesaGuard**, a production-grade real-time fraud detection system. We will perform exploratory data analysis on the PaySim mobile money dataset, implement a custom scikit-learn feature engineering pipeline, and address severe class imbalance.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

sns.set_theme(style='darkgrid')
print('Imports complete!')

## 1. Load Dataset

We configure the notebook to run on either the local 100K sample or the full 6.3M PaySim dataset (e.g. on Google Colab).

In [ ]:
# Toggle this to True if running on Google Colab with the full dataset
USE_FULL_DATASET = False
data_path = 'data/paysim_sample.csv'

if USE_FULL_DATASET:
    data_path = 'PS_20174392719_1491204439457_log.csv'

if not os.path.exists(data_path):
    print(f'Error: Data not found at {data_path}. Please run download_data.py first.')
else:
    df = pd.read_csv(data_path)
    print(f'Loaded dataset from {data_path}')
    print(f'Shape: {df.shape}')

## 2. Exploratory Data Analysis (EDA)

Let's visualize the class distribution, transaction type patterns, amounts, and hourly fraud rates.

In [ ]:
# Class distribution
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100
print('Class Imbalance:')
print(f'Legitimate: {fraud_counts[0]:,} ({fraud_pct[0]:.4f}%)')
print(f'Fraudulent: {fraud_counts[1]:,} ({fraud_pct[1]:.4f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='isFraud', data=df, ax=ax)
ax.set_title('Transaction Class Counts (0 = Legit, 1 = Fraud)')
plt.yscale('log') # Log scale to visualize the tiny fraud class
plt.show()

In [ ]:
# Fraud by Transaction Type
fraud_by_type = df.groupby('type')['isFraud'].agg(['count', 'sum'])
fraud_by_type['rate (%)'] = (fraud_by_type['sum'] / fraud_by_type['count']) * 100
print('Fraud rates by transaction type:')
print(fraud_by_type)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=fraud_by_type.index, y='rate (%)', data=fraud_by_type, ax=ax)
ax.set_title('Fraud Rate (%) by Transaction Type')
plt.show()

In [ ]:
# Amount distribution by class
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x='isFraud', y='amount', data=df, ax=ax)
ax.set_title('Transaction Amounts by Class')
plt.yscale('log')
plt.show()

In [ ]:
# Hourly fraud patterns
df['hour'] = df['step'] % 24
hourly_stats = df.groupby('hour')['isFraud'].agg(['count', 'sum'])
hourly_stats['rate (%)'] = (hourly_stats['sum'] / hourly_stats['count']) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
sns.lineplot(x=hourly_stats.index, y='count', data=hourly_stats, ax=ax1, marker='o', color='blue')
ax1.set_ylabel('Total Volume')
ax1.set_title('Transaction Volume vs. Fraud Rate by Hour of Day')

sns.lineplot(x=hourly_stats.index, y='rate (%)', data=hourly_stats, ax=ax2, marker='s', color='red')
ax2.set_ylabel('Fraud Rate (%)')
ax2.set_xlabel('Hour of Day (step % 24)')

plt.xticks(range(24))
plt.tight_layout()
plt.show()

## 3. Custom Feature Engineering Pipeline

To build a production-grade system, we implement our feature engineering inside a scikit-learn custom transformer. This ensures that we can deploy the pipeline directly to production to score live single transactions. We support both **Offline Batch Mode** (running aggregations on the historical dataframe) and **Online Real-Time Mode** (enriching a single incoming transaction using the customer's state from the database).

In [ ]:
class PesaGuardFeaturePipeline(BaseEstimator, TransformerMixin):
    def __init__(self):
        # For offline training, we will build a history dictionary for each account
        self.user_history_ = {}
        self.merchant_fraud_counts_ = {}
        self.merchant_total_counts_ = {}
        self.merchant_risk_scores_ = {}

    def fit(self, X, y=None):
        # Fit learns history from the training set to prevent leakage
        df_fit = X.copy()
        if y is not None:
            df_fit['isFraud'] = y
        else:
            df_fit['isFraud'] = 0

        # Learn merchant risk scores
        # Destination accounts starting with 'M' are merchants
        df_fit['is_merchant'] = df_fit['nameDest'].str.startswith('M')
        df_fit['dest_prefix'] = df_fit['nameDest'].str[0]

        # Compute fraud counts per destination category
        merchant_stats = df_fit.groupby('dest_prefix')['isFraud'].agg(['count', 'sum'])
        for prefix, row in merchant_stats.iterrows():
            # Frequency-based risk score
            self.merchant_risk_scores_[prefix] = row['sum'] / row['count'] if row['count'] > 0 else 0.0

        # Pre-compute historical statistics for users present in fit data
        # (For offline testing, we sort by step to emulate sequential arrival)
        df_sorted = df_fit.sort_values(by='step')
        for idx, row in df_sorted.iterrows():
            user = row['nameOrig']
            amt = row['amount']
            step = row['step']
            
            if user not in self.user_history_:
                self.user_history_[user] = {'amounts': [], 'steps': []}
            
            self.user_history_[user]['amounts'].append(amt)
            self.user_history_[user]['steps'].append(step)
            
        return self

    def transform(self, X):
        # Transforms data. In production, this can score batched or single rows.
        df_trans = X.copy().sort_values(by='step')
        
        engineered_rows = []
        
        # Temporary history to update sequentially during transformation
        temp_history = {u: {'amounts': list(h['amounts']), 'steps': list(h['steps'])} 
                        for u, h in self.user_history_.items()}
        
        for idx, row in df_trans.iterrows():
            user = row['nameOrig']
            dest = row['nameDest']
            amt = row['amount']
            step = row['step']
            balance_orig = row['oldbalanceOrg']
            
            # 1. Basic features
            hour_of_day = step % 24
            is_round_amount = 1 if (amt % 100 == 0 or amt % 1000 == 0) and amt > 0 else 0
            cross_border = 1 if row['nameDest'].startswith('M') else 0 # simple logic for merchant/external transfer
            amount_to_balance_ratio = amt / (balance_orig + 1e-5) # add small epsilon to prevent div by zero
            
            # Get destination merchant risk score
            dest_prefix = dest[0] if len(dest) > 0 else 'C'
            merchant_risk_score = self.merchant_risk_scores_.get(dest_prefix, 0.0)

            # 2. User history features
            user_history = temp_history.get(user, {'amounts': [], 'steps': []})
            user_amts = user_history['amounts']
            user_steps = user_history['steps']
            
            if len(user_amts) > 0:
                # 30-day average (720 steps)
                recent_amts = [a for a, s in zip(user_amts, user_steps) if step - s <= 720]
                recent_steps = [s for s in user_steps if step - s <= 720]
                
                # Average amount
                avg_amt = np.mean(recent_amts) if len(recent_amts) > 0 else amt
                std_amt = np.std(recent_amts) if len(recent_amts) > 1 else 0.0
                amount_deviation = (amt - avg_amt) / (std_amt + 1e-5) # Z-score
                
                # Velocity
                velocity_1h = sum(1 for s in recent_steps if step - s <= 1)
                velocity_24h = sum(1 for s in recent_steps if step - s <= 24)
                
                # Time since last transaction (step is in hours)
                time_since_last_transaction = step - user_steps[-1]
                
                # Amount percentile in history
                amount_percentile = np.percentile(recent_amts, 50) # simple median reference as fallback
                if len(recent_amts) > 2:
                    amount_percentile = sum(1 for a in recent_amts if amt >= a) / len(recent_amts)
                else:
                    amount_percentile = 0.5
            else:
                # First transaction for this user
                amount_deviation = 0.0
                velocity_1h = 1
                velocity_24h = 1
                time_since_last_transaction = 999.0 # flag for first transaction
                amount_percentile = 0.5
                
            # Record engineered feature row
            feat_row = {
                'amount_deviation': amount_deviation,
                'velocity_1h': velocity_1h,
                'velocity_24h': velocity_24h,
                'amount_to_balance_ratio': amount_to_balance_ratio,
                'hour_of_day': hour_of_day,
                'is_round_amount': is_round_amount,
                'merchant_risk_score': merchant_risk_score,
                'cross_border': cross_border,
                'time_since_last_transaction': time_since_last_transaction,
                'amount_percentile': amount_percentile,
                'amount': amt,
                'oldbalanceOrg': balance_orig,
                'newbalanceOrig': row['newbalanceOrig'],
                'oldbalanceDest': row['oldbalanceDest'],
                'newbalanceDest': row['newbalanceDest'],
                'is_transfer': 1 if row['type'] == 'TRANSFER' else 0,
                'is_cash_out': 1 if row['type'] == 'CASH_OUT' else 0
            }
            engineered_rows.append(feat_row)
            
            # Update history dynamically
            if user not in temp_history:
                temp_history[user] = {'amounts': [], 'steps': []}
            temp_history[user]['amounts'].append(amt)
            temp_history[user]['steps'].append(step)
            
        return pd.DataFrame(engineered_rows)

pipeline = PesaGuardFeaturePipeline()
print('Feature engineering class compiled!')

## 4. Time-Based Splitting & Fitting

Fraud patterns evolve. Random splits cause data leakage (using future data to predict past fraud). We use a chronological split (split by `step`): the first 80% of steps for training, and the final 20% for validation.

In [ ]:
# Chronological split
split_step = int(df['step'].max() * 0.8)
train_df = df[df['step'] <= split_step]
test_df = df[df['step'] > split_step]

X_train, y_train = train_df.drop(columns=['isFraud']), train_df['isFraud']
X_test, y_test = test_df.drop(columns=['isFraud']), test_df['isFraud']

print(f'Training steps: <= {split_step} (Size: {len(X_train)} rows, Fraud cases: {y_train.sum()})')
print(f'Testing steps: > {split_step} (Size: {len(X_test)} rows, Fraud cases: {y_test.sum()})')

# Fit and transform features
pipeline.fit(X_train, y_train)
X_train_trans = pipeline.transform(X_train)
X_test_trans = pipeline.transform(X_test)

print('Features engineered successfully!')
print('Transformed training features columns:')
print(X_train_trans.columns.tolist())
print(X_train_trans.head(3))

## 5. Class Imbalance: SMOTE vs. XGBoost Class Weights

Because our fraud rate is only 0.11%, standard classifiers will simply guess 'legitimate' for all rows. In the next notebook, we will test:
1. **SMOTE** (Synthetic Minority Over-sampling Technique) to generate synthetic fraud samples during training.
2. **scale_pos_weight** in XGBoost to penalize errors on fraud cases proportional to the imbalance.
We will save our engineered training sets for the model notebook.

In [ ]:
# Save processed features for the modeling notebook
if not os.path.exists('data/processed'):
    os.makedirs('data/processed')

X_train_trans.to_csv('data/processed/X_train_feats.csv', index=False)
X_test_trans.to_csv('data/processed/X_test_feats.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)
print('Processed datasets saved for Modeling phase!')